### Đọc dữ liệu

In [45]:
import pandas as pd

df_gmv = pd.read_csv("./cleaned_data/gmv_clean_sales_infor.csv", encoding="utf-8-sig")
df_leads = pd.read_csv("./cleaned_data/leads_clean_with_note_phone.csv", encoding="utf-8-sig")

### Chuẩn hóa cột Phone_clean trong cả 2 bảng

In [46]:
import pandas as pd

def normalize_phone(x):
    if pd.isna(x):
        return ""

    x = str(x).strip()

    if x.endswith(".0"):
        x = x[:-2]

    return x

df_gmv["Phone_clean"] = df_gmv["Phone_clean"].apply(normalize_phone)
df_leads["Phone_clean"] = df_leads["Phone_clean"].apply(normalize_phone)

### Tìm bản match bằng Phone_clean trước, sau đó ưu tiên bởi Sales_infor trong gmv và TVTS trong leads

In [47]:
import numpy as np

# ====================================================
# Chuẩn hóa dữ liệu
# ====================================================

df_gmv = df_gmv.copy()
df_leads = df_leads.copy()

for col in ["Phone_clean", "Sales_infor"]:
    df_gmv[col] = (
        df_gmv[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

for col in ["Phone_clean", "TVTS"]:
    df_leads[col] = (
        df_leads[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# ID để nhận diện từng record GMV
df_gmv["_gmv_id"] = np.arange(len(df_gmv))

# ====================================================
# Join theo Phone_clean
# ====================================================

candidate = df_gmv.merge(
    df_leads,
    on="Phone_clean",
    how="left",
    suffixes=("_gmv", "_lead")
)

# ====================================================
# Ưu tiên Sales_infor == TVTS
# ====================================================

candidate["_priority"] = (
    candidate["Sales_infor"] == candidate["TVTS"]
).astype(int)

# Sắp xếp:
# 1. cùng Sales
# 2. giữ nguyên thứ tự xuất hiện trong leads
candidate = candidate.sort_values(
    by=["_gmv_id", "_priority"],
    ascending=[True, False],
    kind="stable"
)

# ====================================================
# Mỗi GMV chỉ lấy đúng 1 Lead
# ====================================================

matched = (
    candidate
    .drop_duplicates("_gmv_id", keep="first")
    .drop(columns=["_priority"])
)

# ====================================================
# Thống kê
# ====================================================

matched_count = matched["Phone_clean"].ne("").sum() - matched["TVTS"].isna().sum()

print(f"GMV records: {len(df_gmv)}")
print(f"Matched records: {matched['TVTS'].notna().sum()}")
print(f"Unmatched records: {matched['TVTS'].isna().sum()}")

matched.head()

GMV records: 463
Matched records: 281
Unmatched records: 182


,bank day,bank time,Gateway,User Name,Phone,UID_gmv,Package,Fixed/ non-fixed,Pay Time,Real Pay(VND),...,Ngày lên học thử,Current Binded Sale,Sales in latest system-weighted allocation,Latest system-weighted Allocation Time,Allocate reason,Tên sale chatpage,KOL,Check Ad_ID,UID_clean_lead,Phone_extracted_from_note
0,2026/5/2,18:28,VP Bank,C Thanh - Bé Hân,84-908224672,3311001921,2/W- NEW 96 PHI+10,2/W,2026/5/2,18.100.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026/5/11,NaN,NaN,Chị Nga - Bé Khang,84-909517679,3179205818,2/W- NEW 24 PHI+2,2/W,2026/5/11,4.680.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026/5/11,20:33,BIDV,Chị Diễn - Bé Thư Di,84-964678701,3311304274,2/W- NEW 24 PHI+2,2/W,2026/5/11,4.680.000,...,2026-05-11,Lai Ngoc Thuy Linh,Lai Ngoc Thuy Linh,2026-05-09 22:52:34,Weighted,HCM - LinhLNT,NaN,NaN,3311304274,NaN
3,2026/5/14,09:30,Agribank,Chị Huệ - Bé Thi,84-389970625,3310902627,2/W- Both AB (A-PH) REFER 96 PHI+10,2/W,2026/5/14,17.490.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026/5/14,17:59,Techcombank,Chị Ngọc Anh - Bé Nhân,84-938593961,3309271098,2/W- NEW 24 PHI+2,2/W,2026/5/14,4.900.000,...,NaN,Lai Ngoc Thuy Linh,Lai Ngoc Thuy Linh,Lai Ngoc Thuy Linh,Weighted,HCM - LinhLNT,NaN,NaN,3309271098,NaN


In [48]:
# Các GMV đã match thành công
matched_gmv_ids = matched.loc[
    matched["TVTS"].notna(),
    "_gmv_id"
]

# Giữ lại các GMV chưa match
df_gmv_unmatched = df_gmv[
    ~df_gmv["_gmv_id"].isin(matched_gmv_ids)
].copy()

print(f"Matched: {len(matched_gmv_ids)}")
print(f"Remaining unmatched: {len(df_gmv_unmatched)}")

Matched: 281
Remaining unmatched: 182


In [49]:
df_gmv = df_gmv[
    ~df_gmv["_gmv_id"].isin(matched_gmv_ids)
].copy()

In [50]:
print(df_gmv.shape)

(182, 41)


### Sử dụng Phone_extracted_from_note để tiếp tục tìm bản match

In [51]:
df_leads["Phone_extracted_from_note"] = df_leads["Phone_extracted_from_note"].apply(normalize_phone)

In [52]:
df_leads["Phone_extracted_from_note"].head()

0              
1              
2    0965468525
3    0985951781
4              
Name: Phone_extracted_from_note, dtype: object

In [53]:
# ====================================================
# Chuẩn hóa dữ liệu
# ====================================================

df_gmv = df_gmv.copy()
df_leads = df_leads.copy()

for col in ["Phone_clean", "Sales_infor"]:
    df_gmv[col] = (
        df_gmv[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

for col in ["Phone_extracted_from_note", "TVTS"]:
    df_leads[col] = (
        df_leads[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# ID duy nhất cho từng record GMV
df_gmv["_gmv_id"] = np.arange(len(df_gmv))

# ====================================================
# Match Phone_clean (GMV) <-> Phone_extracted_from_note (Leads)
# ====================================================

candidate = df_gmv.merge(
    df_leads,
    left_on="Phone_clean",
    right_on="Phone_extracted_from_note",
    how="left",
    suffixes=("_gmv", "_lead")
)

# ====================================================
# Ưu tiên Sales_infor == TVTS
# ====================================================

candidate["_priority"] = (
    candidate["Sales_infor"] == candidate["TVTS"]
).astype(int)

candidate = candidate.sort_values(
    by=["_gmv_id", "_priority"],
    ascending=[True, False],
    kind="stable"
)

matched = (
    candidate
    .drop_duplicates("_gmv_id", keep="first")
    .drop(columns="_priority")
)

# ====================================================
# Thống kê
# ====================================================

matched_records = matched[matched["TVTS"].notna()]

print(f"Matched: {len(matched_records)}")
print(f"Unmatched: {len(matched) - len(matched_records)}")

Matched: 3
Unmatched: 179


In [54]:
matched_gmv_ids = matched.loc[
    matched["TVTS"].notna(),
    "_gmv_id"
]

df_gmv = df_gmv[
    ~df_gmv["_gmv_id"].isin(matched_gmv_ids)
].copy()

df_gmv.drop(columns="_gmv_id", inplace=True, errors="ignore")

In [55]:
print(df_gmv.shape)

(179, 40)


### Match bằng UID

In [56]:
# df_gmv.to_excel("./output/test.xlsx")

In [57]:
df_gmv["UID_clean"]

0      3311001921
1      3179205818
3      3310902627
9      3311951330
10     3311919278
          ...    
452    3180717063
458    3310177620
460    3311996848
461    3312285639
462    3312285639
Name: UID_clean, Length: 179, dtype: int64

In [58]:
df_gmv.columns

Index(['bank day', 'bank time', 'Gateway', 'User Name', 'Phone', 'UID',
       'Package', 'Fixed/ non-fixed', 'Pay Time', 'Real Pay(VND)', 'GMV (RMB)',
       'Payment Method', 'Type', 'Sales', 'Note', 'Month of payment',
       'Unnamed: 16', 'TEAM', 'Full Price(VND)', '采购价格 \r\n（套餐配置）', '渠道号',
       '首次申请试听日期', '首次申请试听渠道号', '首次购课时间', 'Source', '已批工单', '总 B (被推荐） 课数',
       'A (推荐人） 课数 (送PH)', 'A', 'A (推荐人）课数 (送UK)', 'B', '实际卖课单价 VND',
       '实际卖课单价 RMB', 'PH/UK', '采购价格（包括转介绍赠送课)', 'Nguồn', 'UID_clean',
       'Phone_clean', 'Sales_infor', 'Sales_Abbr'],
      dtype='object')

In [59]:
# ====================================================
# Chuẩn hóa dữ liệu
# ====================================================

df_gmv = df_gmv.copy()
df_leads = df_leads.copy()

for col in ["UID_clean", "Sales_infor"]:
    df_gmv[col] = (
        df_gmv[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

for col in ["UID_clean", "TVTS"]:
    df_leads[col] = (
        df_leads[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# ID duy nhất cho từng record GMV
df_gmv["_gmv_id"] = np.arange(len(df_gmv))

# ====================================================
# Match UID_clean
# ====================================================

candidate = df_gmv.merge(
    df_leads,
    on="UID_clean",
    how="left",
    suffixes=("_gmv", "_lead")
)

# ====================================================
# Ưu tiên Sales_infor == TVTS
# ====================================================

candidate["_priority"] = (
    candidate["Sales_infor"] == candidate["TVTS"]
).astype(int)

candidate = candidate.sort_values(
    by=["_gmv_id", "_priority"],
    ascending=[True, False],
    kind="stable"
)

matched = (
    candidate
    .drop_duplicates("_gmv_id", keep="first")
    .drop(columns="_priority")
)

# ====================================================
# Thống kê
# ====================================================

matched_records = matched[matched["TVTS"].notna()]

print(f"Matched: {len(matched_records)}")
print(f"Unmatched: {len(matched) - len(matched_records)}")

Matched: 0
Unmatched: 179


In [60]:
matched_gmv_ids = matched.loc[
    matched["TVTS"].notna(),
    "_gmv_id"
]

df_gmv = df_gmv[
    ~df_gmv["_gmv_id"].isin(matched_gmv_ids)
].copy()

df_gmv.drop(columns="_gmv_id", inplace=True, errors="ignore")

In [61]:
df_gmv.shape

(179, 40)

### Bỏ các gmv có type là 转介绍, Refer, Offline, Booth

In [62]:
df_gmv["Type"].unique()

array(['Refer', 'GD', '转介绍', '公海', 'Other', 'Offline', '广告', 'KOC',
       'Lives'], dtype=object)

In [63]:
# Chuẩn hóa trước để tránh lỗi do khoảng trắng
df_gmv["Type"] = df_gmv["Type"].fillna("").astype(str).str.strip()

# Loại bỏ các Type không cần
df_gmv = df_gmv[
    ~df_gmv["Type"].isin([
        "转介绍",
        "Refer",
        "Offline",
        "Booth"
    ])
].copy()

print(df_gmv["Type"].value_counts())
print(len(df_gmv))

Type
公海       32
广告       13
Other     5
GD        2
KOC       1
Lives     1
Name: count, dtype: int64
54


In [64]:
df_gmv["Type"].unique()

array(['GD', '公海', 'Other', '广告', 'KOC', 'Lives'], dtype=object)

In [65]:
df_gmv.to_csv("./output/test_2.csv", index=False)

Số đang dùng: 0 giá trị trùng
phone_number: 0 giá trị trùng
Số điện thoại.1: 0 giá trị trùng
Số điện thoại.1: 0 giá trị trùng
Unnamed: 18: 0 giá trị trùng
Unnamed: 18: 0 giá trị trùng
Số điện thoại: 2 giá trị trùng
Ví dụ: ['84909974812', '84986713269']
Số điện thoại: 6 giá trị trùng
Ví dụ: [#'818062508538', '84392218155', #'84933889568', '84944022155', '84966277873', '84974282165']
Sốđiệnthoại: 1 giá trị trùng
Ví dụ: ['84909974812']

In [66]:
df_gmv["Sales_infor"].unique()

array(['HCM - LinhLNT', 'HN - NgaNTH', 'HN - PhuongTTN', 'HN - AnhLTL',
       'HN2 - HienVTK', 'HN - ThaoPT', 'HN - ThanhHT', 'HN - NgocNTT',
       'HN - AnhNT', 'HN - NhungLT', 'HN2 - LienMT', 'HN - ChiLK',
       'HN - SonTT', 'HN - LuaCT', 'HN - QuynhKTT', 'OFF - Julia',
       'HN2 - HaTT', 'HN - TrangNT', 'HN - TuNC', 'HN2 - HuongVHT',
       'HN - NhungDTT', 'HN2 - HuyenNTT', 'HN - NgocNN', 'HN - HuyenDMN',
       'HN - MyLTH', 'HN2 - TrangNT', 'HN - TrangNK', 'HN - DuongBTT',
       'HN - ThuyPTT', 'HN - YenLTT', 'HN - ThuongDK'], dtype=object)

In [67]:
df_tt_check = pd.read_csv("./data_test/PalFish - Chatpage 2026 - TT check.csv", encoding='utf-8')
df_active_lc = pd.read_csv("./data_test/PalFish - Chatpage 2026 - Active LC.csv", encoding='utf-8')
df_tt_110 = pd.read_csv("./data_test/PalFish - Chatpage 2026 - Trang tính110.csv", encoding='utf-8')

In [68]:
df_gmv["Phone"]

1        84-909517679
11       84-909283440
38     49-15205315239
43       84-963971737
44      82-1057761801
52       84-842625089
106      84-918658586
110      84-366790797
113      84-986713269
115      84-987415373
119     81-9080541507
122      84-943728043
135      84-971525303
163      84-966277873
204     82-1043068055
207      84-905263277
209      84-392218155
212      84-902105222
213      84-909974812
216      84-349544059
219      84-966155489
225      84-963881664
229      84-987189666
232      84-389513868
235      84-909400611
237      84-988464337
239      84-978026892
253      84-964166015
257     81-9042332139
268      84-911331626
269      84-373406103
278      84-986203686
280      84-366918069
289      84-938297945
290      84-975567075
304      84-846028989
315     82-1029349211
316     82-1029349212
319      84-387996228
323      84-907005400
332      84-981228169
334     421-949326547
354      84-904826780
358      84-906764213
363     81-8045432624
371      8

In [69]:
import re
def normalize_phone(phone):
    if pd.isna(phone):
        return ""

    phone = str(phone).strip()

    # Bỏ .0 nếu có
    if phone.endswith(".0"):
        phone = phone[:-2]

    # Chỉ giữ lại chữ số
    phone = re.sub(r"\D", "", phone)

    return phone

In [70]:
df_tt_check["Phone_clean"] = df_tt_check["Số điện thoại"].apply(normalize_phone)

df_active_lc["Phone_clean"] = df_active_lc["Số điện thoại"].apply(normalize_phone)

df_tt_110["Phone_clean"] = df_tt_110["Sốđiệnthoại"].apply(normalize_phone)

In [71]:
df_tt_check["Phone_clean"]
df_active_lc["Phone_clean"]
df_tt_110["Phone_clean"]

0       84357007993
1       84974563233
2       84934049510
3       84869273022
4       84913833004
           ...     
5548               
5549               
5550               
5551               
5552               
Name: Phone_clean, Length: 5553, dtype: object

In [72]:
def remove_matched_gmv(df_gmv, df_other, phone_col="Phone_clean", table_name=""):
    # Chuẩn hóa Phone_clean
    gmv_phone = (
        df_gmv["Phone_clean"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    other_phone = (
        df_other[phone_col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # Tập phone của bảng còn lại
    phone_set = set(other_phone)
    phone_set.discard("")   # bỏ chuỗi rỗng nếu có

    # Những record GMV match
    mask = gmv_phone.isin(phone_set)

    matched_count = mask.sum()

    print(f"{table_name:<15} Matched: {matched_count:>5} | Remaining: {(~mask).sum():>5}")

    # Trả về các record chưa match
    return df_gmv.loc[~mask].copy()

In [73]:
print(f"Initial GMV: {len(df_gmv)}")

df_gmv = remove_matched_gmv(
    df_gmv,
    df_tt_check,
    table_name="TT Check"
)

df_gmv = remove_matched_gmv(
    df_gmv,
    df_active_lc,
    table_name="Active LC"
)

df_gmv = remove_matched_gmv(
    df_gmv,
    df_tt_110,
    table_name="TT 110"
)

print(f"\nFinal remaining GMV: {len(df_gmv)}")

Initial GMV: 54
TT Check        Matched:     2 | Remaining:    52
Active LC       Matched:     2 | Remaining:    50
TT 110          Matched:     0 | Remaining:    50

Final remaining GMV: 50


In [74]:
df_gmv.shape

(50, 40)

In [75]:
df_gmv.to_csv("./output/test_3.csv", index=False)

In [76]:
df_gmv[["User Name", "Sales_infor"]]

,User Name,Sales_infor
1,Chị Nga - Bé Khang,HCM - LinhLNT
11,C Vy - Hải Đăng,HCM - LinhLNT
38,bảo nam,HN - NgaNTH
43,Bình An,HN - PhuongTTN
44,Suchin,HN - AnhLTL
52,Phương Linh,HN2 - HienVTK
106,Minh Quang,HN - ThaoPT
110,Minh Hiiếu,HN - ThanhHT
115,Hoàng Sơn,HN - AnhNT
119,Yo,HN - NhungLT


In [77]:
df_gmv["Phone_clean"]

1        84909517679
11       84909283440
38     4915205315239
43       84963971737
44      821057761801
52       84842625089
106      84918658586
110      84366790797
115      84987415373
119     819080541507
122      84943728043
135      84971525303
204     821043068055
207      84905263277
212      84902105222
216      84349544059
219      84966155489
225      84963881664
229      84987189666
232      84389513868
235      84909400611
237      84988464337
239      84978026892
253      84964166015
257     819042332139
268      84911331626
269      84373406103
278      84986203686
280      84366918069
289      84938297945
290      84975567075
304      84846028989
315     821029349211
316     821029349212
319      84387996228
323      84907005400
332      84981228169
334     421949326547
354      84904826780
358      84906764213
363     818045432624
371      84972275866
380      84385106511
385      84989492127
424      84385654216
428      84356689130
439      84932697890
451      8481

In [78]:
df_gmv["Place"] = (
    df_gmv["Phone_clean"]
    .fillna("")
    .astype(str)
    .str.startswith("84")
    .map({True: "VN", False: "OV"})
)

In [79]:
df_gmv[["Place", "Phone_clean"]]

,Place,Phone_clean
1,VN,84909517679
11,VN,84909283440
38,OV,4915205315239
43,VN,84963971737
44,OV,821057761801
52,VN,84842625089
106,VN,84918658586
110,VN,84366790797
115,VN,84987415373
119,OV,819080541507


In [80]:
df_gmv[["User Name", "Sales_infor", "Sales_Abbr"]]

,User Name,Sales_infor,Sales_Abbr
1,Chị Nga - Bé Khang,HCM - LinhLNT,LinhLNT
11,C Vy - Hải Đăng,HCM - LinhLNT,LinhLNT
38,bảo nam,HN - NgaNTH,NgaNTH
43,Bình An,HN - PhuongTTN,PhuongTTN
44,Suchin,HN - AnhLTL,AnhLTL
52,Phương Linh,HN2 - HienVTK,HienVTK
106,Minh Quang,HN - ThaoPT,ThaoPT
110,Minh Hiiếu,HN - ThanhHT,ThanhHT
115,Hoàng Sơn,HN - AnhNT,AnhNT
119,Yo,HN - NhungLT,NhungLT


In [81]:


# ==========================
# Chuẩn hóa chuỗi
# ==========================

def normalize_text(s):
    return (
        s.fillna("")
         .astype(str)
         .str.strip()
    )

df_gmv = df_gmv.copy()
df_leads = df_leads.copy()

df_gmv["User Name"] = normalize_text(df_gmv["User Name"])
df_gmv["Sales_infor"] = normalize_text(df_gmv["Sales_infor"])
df_gmv["Sales_Abbr"] = normalize_text(df_gmv["Sales_Abbr"])

df_leads["Họ và tên"] = normalize_text(df_leads["Họ và tên"])
df_leads["TVTS"] = normalize_text(df_leads["TVTS"])

# ==================================================
# Tạo map: (Họ và tên, TVTS) -> index đầu tiên của df_leads
# ==================================================

lead_map = {}

for idx, row in df_leads.iterrows():
    key = (row["Họ và tên"], row["TVTS"])
    if key not in lead_map:
        lead_map[key] = idx

# ==================================================
# Bước 1: (User Name, Sales_infor)
# ==================================================

gmv_pairs = list(
    zip(
        df_gmv["User Name"],
        df_gmv["Sales_infor"]
    )
)

lead_idx_step1 = [lead_map.get(pair) for pair in gmv_pairs]

mask1 = pd.Series(
    [idx is not None for idx in lead_idx_step1],
    index=df_gmv.index
)

matched_step1 = df_gmv.loc[mask1].copy()

matched_leads_step1 = df_leads.loc[
    [idx for idx in lead_idx_step1 if idx is not None]
].copy()

unmatched = df_gmv.loc[~mask1].copy()

print(f"Step 1 matched: {len(matched_step1)}")
print(f"Remaining: {len(unmatched)}")

# ==================================================
# Bước 2: (User Name, Sales_Abbr)
# ==================================================

gmv_pairs2 = list(
    zip(
        unmatched["User Name"],
        unmatched["Sales_Abbr"]
    )
)

lead_idx_step2 = [lead_map.get(pair) for pair in gmv_pairs2]

mask2 = pd.Series(
    [idx is not None for idx in lead_idx_step2],
    index=unmatched.index
)

matched_step2 = unmatched.loc[mask2].copy()

matched_leads_step2 = df_leads.loc[
    [idx for idx in lead_idx_step2 if idx is not None]
].copy()

df_gmv = unmatched.loc[~mask2].copy()

print(f"Step 2 matched: {len(matched_step2)}")
print(f"Final remaining: {len(df_gmv)}")

# ==================================================
# Tổng hợp
# ==================================================

matched_all = pd.concat(
    [matched_step1, matched_step2],
    ignore_index=True
)

matched_leads = pd.concat(
    [matched_leads_step1, matched_leads_step2],
    ignore_index=True
)

print(f"\nTotal matched: {len(matched_all)}")
print(f"Matched leads: {len(matched_leads)}")
print(f"Remaining GMV: {len(df_gmv)}")

Step 1 matched: 2
Remaining: 48
Step 2 matched: 0
Final remaining: 48

Total matched: 2
Matched leads: 2
Remaining GMV: 48


In [82]:
matched_all.iloc[0]

bank day                     2026/5/20
bank time                        14:05
Gateway                  Baoviet Smart
User Name                        Khang
Phone                     84-966155489
UID                         3311508463
Package              2/W- NEW 48 PHI+5
Fixed/ non-fixed                   2/W
Pay Time                     2026/5/20
Real Pay(VND)                  9580000
GMV (RMB)                         2589
Payment Method                     1st
Type                                广告
Sales               Kieu Thi Thu Quynh
Note                               NaN
Month of payment                2026/5
Unnamed: 16                        NaN
TEAM                          In-house
Full Price(VND)                    NaN
采购价格 \r\n（套餐配置）                    NaN
渠道号                                NaN
首次申请试听日期                           NaN
首次申请试听渠道号                          NaN
首次购课时间                             NaN
Source                             NaN
已批工单                     

In [83]:
matched_leads.iloc[0]

Đầu Ngày                                                                                  21/02
Họ và tên                                                                                 Khang
Người trực                                                                                  NaN
Số điện thoại                                                                     81-9092683838
Tuổi                                                                                        NaN
Quốc Gia                                                                               Nhật Bản
Note                                          mẫu_giáo_(3t-5t)_        yêu_thích_và_có_khả_n...
TVTS                                                                              HN - QuynhKTT
Ad_id                                                                    1.2024154625953056e+17
Nguồn                                                                                       NaN
Quốc gia \nQuảng cáo                    

In [84]:
df_gmv.shape

(48, 41)

In [85]:
df_gmv["Place"]

1      VN
11     VN
38     OV
43     VN
44     OV
52     VN
106    VN
110    VN
115    VN
119    OV
122    VN
135    VN
204    OV
207    VN
212    VN
216    VN
225    VN
232    VN
235    VN
237    VN
239    VN
253    VN
257    OV
268    VN
269    VN
278    VN
280    VN
289    VN
290    VN
304    VN
315    OV
316    OV
319    VN
323    VN
332    VN
334    OV
354    VN
358    VN
363    OV
371    VN
380    VN
385    VN
424    VN
428    VN
439    VN
451    VN
461    VN
462    VN
Name: Place, dtype: object

In [86]:
df_gmv.head(10)

,bank day,bank time,Gateway,User Name,Phone,UID,Package,Fixed/ non-fixed,Pay Time,Real Pay(VND),...,实际卖课单价 VND,实际卖课单价 RMB,PH/UK,采购价格（包括转介绍赠送课),Nguồn,UID_clean,Phone_clean,Sales_infor,Sales_Abbr,Place
1,2026/5/11,NaN,NaN,Chị Nga - Bé Khang,84-909517679,3179205818,2/W- NEW 24 PHI+2,2/W,2026/5/11,4.680.000,...,NaN,NaN,NaN,NaN,NaN,3179205818,84909517679,HCM - LinhLNT,LinhLNT,VN
11,2026/5/28,10:59,VCB,C Vy - Hải Đăng,84-909283440,3302992772,2/W- NEW 24 PHI+2,2/W,2026/5/28,4420000,...,NaN,NaN,NaN,NaN,NaN,3302992772,84909283440,HCM - LinhLNT,LinhLNT,VN
38,2026/5/5,21:35,Agribank,bảo nam,49-15205315239,3287789579,2/W- NEW 48 PHI+5,2/W,2026/5/5,10080000,...,"190,189",51.0,PH,NaN,NaN,3287789579,4915205315239,HN - NgaNTH,NgaNTH,OV
43,2026/5/5,14:58,Payoo,Bình An,84-963971737,3299808858,2/W- NEW 48 PHI+5,2/W,2026/5/5,9376620,...,"176,917",48.0,PH,NaN,NaN,3299808858,84963971737,HN - PhuongTTN,PhuongTTN,VN
44,2026/5/5,15:37,Techcombank,Suchin,82-1057761801,3267891426,2/W- NEW 48 PHI+5,2/W,2026/5/5,9080000,...,"171,321",46.0,PH,NaN,NaN,3267891426,821057761801,HN - AnhLTL,AnhLTL,OV
52,2026/5/6,15:54,VCB Digibank,Phương Linh,84-842625089,3311032514,2/W- NEW 48 PHI+5,2/W,2026/5/6,9080000,...,"171,321",46.0,PH,NaN,NaN,3311032514,84842625089,HN2 - HienVTK,HienVTK,VN
106,2026/5/11,13:46,VCB Digibank,Minh Quang,84-918658586,3298993560,2/W- NEW 48 PHI+5,2/W,2026/5/11,9600000,...,"181,132",49.0,PH,NaN,NaN,3298993560,84918658586,HN - ThaoPT,ThaoPT,VN
110,2026/5/12,11:02,Techcombank,Minh Hiiếu,84-366790797,3311153969,2/W- NEW 24 PHI+2,2/W,2026/5/12,4790000,...,"184,231",50.0,PH,NaN,NaN,3311153969,84366790797,HN - ThanhHT,ThanhHT,VN
115,2026/5/12,x,VPBank,Hoàng Sơn,84-987415373,3311199404,2/W- NEW 48 PHI+5,2/W,2026/5/12,10080000,...,"190,189",51.0,PH,NaN,NaN,3311199404,84987415373,HN - AnhNT,AnhNT,VN
119,2026/5/12,18:10,MBBank,Yo,81-9080541507,3280414186,2/W- NEW 144 PHI+15 (HN 河内),2/W,2026/5/12,24720000,...,"155,472",42.0,PH,NaN,NaN,3280414186,819080541507,HN - NhungLT,NhungLT,OV


In [87]:
# Count the frequency of each unique value in the 'Place' column
place_counts = df_gmv['Place'].value_counts()

# Display the result
print(place_counts)

Place
VN    39
OV     9
Name: count, dtype: int64


In [88]:
df_gmv.to_csv("./excel_check/48_case.csv", index=False)

OSError: Cannot save file into a non-existent directory: 'excel_check'

In [ ]:
print(df_gmv["Type"].value_counts())

Type
公海       27
广告       12
Other     5
GD        2
KOC       1
Lives     1
Name: count, dtype: int64


In [ ]:
# Create a set of unique values from df_leads["TVTS"] (optimized for speed)
# Drop NaN and strip spaces to ensure accurate matching
leads_set = set(df_leads["TVTS"].dropna().astype(str).str.strip())

# Check which rows in df_gmv["Sales_infor"] are in that set
# We also clean the df_gmv side to match correctly
matching_rows = df_gmv["Sales_infor"].dropna().astype(str).str.strip().isin(leads_set)

# Count how many rows returned True
total_matching_rows = matching_rows.sum()

print(f"Có tổng cộng {total_matching_rows} hàng trong df_gmv['Sales_infor'] có giá trị xuất hiện trong df_leads['TVTS']")

Có tổng cộng 43 hàng trong df_gmv['Sales_infor'] có giá trị xuất hiện trong df_leads['TVTS']


In [ ]:
# 1. Create a clean set of unique values from df_leads["TVTS"]
leads_set = set(df_leads["TVTS"].dropna().astype(str).str.strip())

# 2. Create a clean version of df_gmv["Sales_infor"] to match with the set
# (We use .map() here to keep the index aligned perfectly with df_gmv)
condition = df_gmv["Sales_infor"].astype(str).str.strip().isin(leads_set)

# 3. Filter df_gmv based on the condition
df_matching_rows = df_gmv[condition]

# Display the first few rows of the result
print(f"Đã lấy ra {len(df_matching_rows)} dòng thỏa mãn điều kiện.")
df_matching_rows.head()

Đã lấy ra 43 dòng thỏa mãn điều kiện.


,bank day,bank time,Gateway,User Name,Phone,UID,Package,Fixed/ non-fixed,Pay Time,Real Pay(VND),...,实际卖课单价 VND,实际卖课单价 RMB,PH/UK,采购价格（包括转介绍赠送课),Nguồn,UID_clean,Phone_clean,Sales_infor,Sales_Abbr,Place
1,2026/5/11,NaN,NaN,Chị Nga - Bé Khang,84-909517679,3179205818,2/W- NEW 24 PHI+2,2/W,2026/5/11,4.680.000,...,NaN,NaN,NaN,NaN,NaN,3179205818,84909517679,HCM - LinhLNT,LinhLNT,VN
11,2026/5/28,10:59,VCB,C Vy - Hải Đăng,84-909283440,3302992772,2/W- NEW 24 PHI+2,2/W,2026/5/28,4420000,...,NaN,NaN,NaN,NaN,NaN,3302992772,84909283440,HCM - LinhLNT,LinhLNT,VN
38,2026/5/5,21:35,Agribank,bảo nam,49-15205315239,3287789579,2/W- NEW 48 PHI+5,2/W,2026/5/5,10080000,...,"190,189",51.0,PH,NaN,NaN,3287789579,4915205315239,HN - NgaNTH,NgaNTH,OV
43,2026/5/5,14:58,Payoo,Bình An,84-963971737,3299808858,2/W- NEW 48 PHI+5,2/W,2026/5/5,9376620,...,"176,917",48.0,PH,NaN,NaN,3299808858,84963971737,HN - PhuongTTN,PhuongTTN,VN
44,2026/5/5,15:37,Techcombank,Suchin,82-1057761801,3267891426,2/W- NEW 48 PHI+5,2/W,2026/5/5,9080000,...,"171,321",46.0,PH,NaN,NaN,3267891426,821057761801,HN - AnhLTL,AnhLTL,OV


In [ ]:
df_leads["Phone_clean"].head()

0               
1    84919249237
2    84965468525
3    84985951781
4    84972856556
Name: Phone_clean, dtype: object

In [ ]:
df_leads["Place"] = (
    df_leads["Phone_clean"]
    .fillna("")
    .astype(str)
    .str.startswith("84")
    .map({True: "VN", False: "OV"})
)

In [ ]:
df_leads["Place"]

0        OV
1        VN
2        VN
3        VN
4        VN
         ..
16331    VN
16332    VN
16333    VN
16334    VN
16335    VN
Name: Place, Length: 16336, dtype: object

In [ ]:
df_leads["Đầu Ngày"] = (
    df_leads["Đầu Ngày"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_leads["Đầu Ngày"] = pd.to_datetime(
    df_leads["Đầu Ngày"] + "/2026",
    format="%d/%m/%Y",
    errors="coerce"
)

df_gmv["Pay Time"] = pd.to_datetime(
    df_gmv["Pay Time"],
    format="%Y/%m/%d",
    errors="coerce"
)
# ==========================
# Chuẩn hóa
# ==========================

def normalize_text(s):
    return (
        s.fillna("")
         .astype(str)
         .str.strip()
    )

df_gmv = df_gmv.copy()
df_leads = df_leads.copy()

df_gmv["Sales_infor"] = normalize_text(df_gmv["Sales_infor"])
df_gmv["Place"] = normalize_text(df_gmv["Place"])

df_leads["TVTS"] = normalize_text(df_leads["TVTS"])
df_leads["Place"] = normalize_text(df_leads["Place"])

# ==========================
# Chuẩn hóa datetime
# ==========================

df_gmv["Pay Time"] = pd.to_datetime(
    df_gmv["Pay Time"],
    errors="coerce"
)

df_leads["Đầu Ngày"] = pd.to_datetime(
    df_leads["Đầu Ngày"],
    errors="coerce"
)

# ==================================================
# Map (TVTS, Place) -> list index của leads
# ==================================================

lead_map = {}

for idx, row in df_leads.iterrows():

    key = (
        row["TVTS"],
        row["Place"]
    )

    lead_map.setdefault(key, []).append(idx)

# ==================================================
# Match
# ==================================================

gmv_pairs = list(
    zip(
        df_gmv["Sales_infor"],
        df_gmv["Place"]
    )
)

lead_idx = []

for pair, pay_time in zip(gmv_pairs, df_gmv["Pay Time"]):

    idx_found = None

    if pair in lead_map:

        # duyệt theo đúng thứ tự trong df_leads
        for idx in lead_map[pair]:

            lead_time = df_leads.at[idx, "Đầu Ngày"]

            if pd.notna(lead_time) and pd.notna(pay_time):

                if lead_time <= pay_time:
                    idx_found = idx
                    break

    lead_idx.append(idx_found)

# ==================================================
# Kết quả
# ==================================================

mask = pd.Series(
    [idx is not None for idx in lead_idx],
    index=df_gmv.index
)

# GMV đã match
matched = df_gmv.loc[mask].copy()

# Lead tương ứng
matched_leads = df_leads.loc[
    [idx for idx in lead_idx if idx is not None]
].copy()

# Loại khỏi GMV
df_gmv = df_gmv.loc[~mask].copy()

print("Matched:", len(matched))
print("Matched leads:", len(matched_leads))
print("Remaining GMV:", len(df_gmv))

Matched: 41
Matched leads: 41
Remaining GMV: 7


In [ ]:
print(df_gmv["Pay Time"].head(20))
print(df_gmv["Pay Time"].dtype)

52    2026-05-06
253   2026-05-22
268   2026-05-23
269   2026-05-23
304   2026-05-25
461   2026-05-31
462   2026-05-31
Name: Pay Time, dtype: datetime64[ns]
datetime64[ns]


In [ ]:
df_gmv.head(10)

,bank day,bank time,Gateway,User Name,Phone,UID,Package,Fixed/ non-fixed,Pay Time,Real Pay(VND),...,实际卖课单价 VND,实际卖课单价 RMB,PH/UK,采购价格（包括转介绍赠送课),Nguồn,UID_clean,Phone_clean,Sales_infor,Sales_Abbr,Place
52,2026/5/6,15:54,VCB Digibank,Phương Linh,84-842625089,3311032514,2/W- NEW 48 PHI+5,2/W,2026-05-06,9080000,...,"171,321",46.0,PH,NaN,NaN,3311032514,84842625089,HN2 - HienVTK,HienVTK,VN
253,2026/5/22,x,VIMO,Hồng Minh,84-964166015,3310392292,2/W- NEW 48 PHI+5,2/W,2026-05-22,9212425,...,"173,819",47.0,PH,NaN,NaN,3310392292,84964166015,HN2 - HuongVHT,HuongVHT,VN
268,2026/5/23,21:26,Agribank,Anh Thảo,84-911331626,3271847872,2/W- NEW 96 PHI+10,2/W,2026-05-23,16960000,...,160,43.0,PH,NaN,NaN,3271847872,84911331626,HN2 - HuyenNTT,HuyenNTT,VN
269,2026/5/23,22:55,BIDV,Bảo An,84-373406103,3311874241,2/W- NEW 24 PHI+2,2/W,2026-05-23,4420000,...,170,46.0,PH,NaN,NaN,3311874241,84373406103,HN2 - HuyenNTT,HuyenNTT,VN
304,2026/5/25,19:45,MBBank,Tuấn,84-846028989,3303143877,2/W- NEW 48 PHI+5,2/W,2026-05-25,9010000,...,170,46.0,PH,NaN,Tiktokshop,3303143877,84846028989,HN2 - HienVTK,HienVTK,VN
461,2026/5/31,23:38,x,Thiên Kim,84-943111987,3312285639,2/W- NEW 24 PHI+2,2/W,2026-05-31,4550000,...,175,47.0,PH,NaN,NaN,3312285639,84943111987,HN2 - HuongVHT,HuongVHT,VN
462,2026/5/31,23:38,MBBank,Mon,84-943111987,3312285639,2/W- NEW 24 PHI+2,2/W,2026-05-31,4420000,...,170,46.0,PH,NaN,NaN,3312285639,84943111987,HN2 - HuongVHT,HuongVHT,VN


In [ ]:
df_matching_rows.head()

,bank day,bank time,Gateway,User Name,Phone,UID,Package,Fixed/ non-fixed,Pay Time,Real Pay(VND),...,实际卖课单价 VND,实际卖课单价 RMB,PH/UK,采购价格（包括转介绍赠送课),Nguồn,UID_clean,Phone_clean,Sales_infor,Sales_Abbr,Place
1,2026/5/11,NaN,NaN,Chị Nga - Bé Khang,84-909517679,3179205818,2/W- NEW 24 PHI+2,2/W,2026/5/11,4.680.000,...,NaN,NaN,NaN,NaN,NaN,3179205818,84909517679,HCM - LinhLNT,LinhLNT,VN
11,2026/5/28,10:59,VCB,C Vy - Hải Đăng,84-909283440,3302992772,2/W- NEW 24 PHI+2,2/W,2026/5/28,4420000,...,NaN,NaN,NaN,NaN,NaN,3302992772,84909283440,HCM - LinhLNT,LinhLNT,VN
38,2026/5/5,21:35,Agribank,bảo nam,49-15205315239,3287789579,2/W- NEW 48 PHI+5,2/W,2026/5/5,10080000,...,"190,189",51.0,PH,NaN,NaN,3287789579,4915205315239,HN - NgaNTH,NgaNTH,OV
43,2026/5/5,14:58,Payoo,Bình An,84-963971737,3299808858,2/W- NEW 48 PHI+5,2/W,2026/5/5,9376620,...,"176,917",48.0,PH,NaN,NaN,3299808858,84963971737,HN - PhuongTTN,PhuongTTN,VN
44,2026/5/5,15:37,Techcombank,Suchin,82-1057761801,3267891426,2/W- NEW 48 PHI+5,2/W,2026/5/5,9080000,...,"171,321",46.0,PH,NaN,NaN,3267891426,821057761801,HN - AnhLTL,AnhLTL,OV


In [ ]:
def normalize_text(s):
    return (
        s.fillna("")
         .astype(str)
         .str.strip()
    )

df_gmv = df_gmv.copy()
df_leads = df_leads.copy()

df_gmv["Sales_infor"] = normalize_text(df_gmv["Sales_infor"])
df_gmv["Place"] = normalize_text(df_gmv["Place"])

df_leads["TVTS"] = normalize_text(df_leads["TVTS"])
df_leads["Place"] = normalize_text(df_leads["Place"])

# ==================================================
# Map (TVTS, Place) -> index đầu tiên của lead
# ==================================================

lead_map = {}

for idx, row in df_leads.iterrows():
    key = (row["TVTS"], row["Place"])

    if key not in lead_map:
        lead_map[key] = idx

# ==================================================
# Match
# ==================================================

gmv_pairs = list(
    zip(
        df_gmv["Sales_infor"],
        df_gmv["Place"]
    )
)

lead_idx = [lead_map.get(pair) for pair in gmv_pairs]

mask = pd.Series(
    [idx is not None for idx in lead_idx],
    index=df_gmv.index
)

# GMV đã match
matched = df_gmv.loc[mask].copy()

# Lead tương ứng
matched_leads = df_leads.loc[
    [idx for idx in lead_idx if idx is not None]
].copy()

# Loại khỏi GMV
df_gmv = df_gmv.loc[~mask].copy()

print("Matched:", len(matched))
print("Matched leads:", len(matched_leads))
print("Remaining GMV:", len(df_gmv))

Matched: 2
Matched leads: 2
Remaining GMV: 5


In [ ]:
df_gmv.head()

,bank day,bank time,Gateway,User Name,Phone,UID,Package,Fixed/ non-fixed,Pay Time,Real Pay(VND),...,实际卖课单价 VND,实际卖课单价 RMB,PH/UK,采购价格（包括转介绍赠送课),Nguồn,UID_clean,Phone_clean,Sales_infor,Sales_Abbr,Place
52,2026/5/6,15:54,VCB Digibank,Phương Linh,84-842625089,3311032514,2/W- NEW 48 PHI+5,2/W,2026-05-06,9080000,...,"171,321",46.0,PH,NaN,NaN,3311032514,84842625089,HN2 - HienVTK,HienVTK,VN
253,2026/5/22,x,VIMO,Hồng Minh,84-964166015,3310392292,2/W- NEW 48 PHI+5,2/W,2026-05-22,9212425,...,"173,819",47.0,PH,NaN,NaN,3310392292,84964166015,HN2 - HuongVHT,HuongVHT,VN
304,2026/5/25,19:45,MBBank,Tuấn,84-846028989,3303143877,2/W- NEW 48 PHI+5,2/W,2026-05-25,9010000,...,170,46.0,PH,NaN,Tiktokshop,3303143877,84846028989,HN2 - HienVTK,HienVTK,VN
461,2026/5/31,23:38,x,Thiên Kim,84-943111987,3312285639,2/W- NEW 24 PHI+2,2/W,2026-05-31,4550000,...,175,47.0,PH,NaN,NaN,3312285639,84943111987,HN2 - HuongVHT,HuongVHT,VN
462,2026/5/31,23:38,MBBank,Mon,84-943111987,3312285639,2/W- NEW 24 PHI+2,2/W,2026-05-31,4420000,...,170,46.0,PH,NaN,NaN,3312285639,84943111987,HN2 - HuongVHT,HuongVHT,VN


In [ ]:
def get_series_intersection(series_1: pd.Series, series_2: pd.Series) -> set:
    """
    Find the intersection of two Pandas Series (columns) safely.
    
    Parameters:
    series_1 (pd.Series): The first column (e.g., df_gmv["ColumnA"])
    series_2 (pd.Series): The second column (e.g., df_leads["ColumnB"])
    
    Returns:
    set: A set containing the overlapping string values.
    """
    # Clean and convert the first column to a set
    set_1 = set(series_1.dropna().astype(str).str.strip())
    
    # Clean and convert the second column to a set
    set_2 = set(series_2.dropna().astype(str).str.strip())
    
    # Find the common values
    intersection_result = set_1.intersection(set_2)
    
    # Print execution summary
    print(f"--- Intersection Summary ---")
    print(f"Unique values in Series 1: {len(set_1)}")
    print(f"Unique values in Series 2: {len(set_2)}")
    print(f"Overlapping values count : {len(intersection_result)}\n")
    
    return intersection_result

# Call the function by passing your two columns
common_values = get_series_intersection(df_gmv["User Name"], df_leads["Họ và tên"])

# Display the intersecting elements
print("Intersecting values:", common_values)

--- Intersection Summary ---
Unique values in Series 1: 3
Unique values in Series 2: 12719
Overlapping values count : 0

Intersecting values: set()


In [ ]:
# ==========================
# Chuẩn hóa
# ==========================

def normalize_text(s):
    return (
        s.fillna("")
         .astype(str)
         .str.strip()
    )

df_gmv = df_gmv.copy()
df_leads = df_leads.copy()

df_gmv["User Name"] = normalize_text(df_gmv["User Name"])
df_gmv["Place"] = normalize_text(df_gmv["Place"])

df_leads["Họ và tên"] = normalize_text(df_leads["Họ và tên"])
df_leads["Place"] = normalize_text(df_leads["Place"])

# ==========================
# Chuẩn hóa ngày
# ==========================

df_gmv["Pay Time"] = pd.to_datetime(
    df_gmv["Pay Time"],
    format="%Y/%m/%d",
    errors="coerce"
)

df_leads["Đầu Ngày"] = (
    df_leads["Đầu Ngày"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_leads["Đầu Ngày"] = pd.to_datetime(
    df_leads["Đầu Ngày"] + "/2026",
    format="%d/%m/%Y",
    errors="coerce"
)

# ==================================================
# Map (Họ và tên, Place) -> list index
# ==================================================

lead_map = {}

for idx, row in df_leads.iterrows():

    key = (
        row["Họ và tên"],
        row["Place"]
    )

    lead_map.setdefault(key, []).append(idx)

# ==================================================
# Match
# ==================================================

gmv_pairs = list(
    zip(
        df_gmv["User Name"],
        df_gmv["Place"]
    )
)

lead_idx = []

for pair, pay_time in zip(gmv_pairs, df_gmv["Pay Time"]):

    idx_found = None

    if pair in lead_map:

        for idx in lead_map[pair]:

            lead_time = df_leads.at[idx, "Đầu Ngày"]

            if pd.notna(lead_time) and pd.notna(pay_time):

                if lead_time <= pay_time:
                    idx_found = idx
                    break

    lead_idx.append(idx_found)

# ==================================================
# Kết quả
# ==================================================

mask = pd.Series(
    [idx is not None for idx in lead_idx],
    index=df_gmv.index
)

matched = df_gmv.loc[mask].copy()

matched_leads = df_leads.loc[
    [idx for idx in lead_idx if idx is not None]
].copy()

df_gmv = df_gmv.loc[~mask].copy()

print("Matched:", len(matched))
print("Matched leads:", len(matched_leads))
print("Remaining GMV:", len(df_gmv))

Matched: 0
Matched leads: 0
Remaining GMV: 5


In [ ]:
# ==========================
# Chuẩn hóa
# ==========================

def normalize_text(s):
    return (
        s.fillna("")
         .astype(str)
         .str.strip()
    )

df_gmv = df_gmv.copy()
df_leads = df_leads.copy()

df_gmv["User Name"] = normalize_text(df_gmv["User Name"])
df_gmv["Place"] = normalize_text(df_gmv["Place"])

df_leads["Họ và tên"] = normalize_text(df_leads["Họ và tên"])
df_leads["Place"] = normalize_text(df_leads["Place"])

# ==================================================
# Map (Họ và tên, Place) -> index đầu tiên của lead
# ==================================================

lead_map = {}

for idx, row in df_leads.iterrows():
    key = (
        row["Họ và tên"],
        row["Place"]
    )

    if key not in lead_map:
        lead_map[key] = idx

# ==================================================
# Match
# ==================================================

gmv_pairs = list(
    zip(
        df_gmv["User Name"],
        df_gmv["Place"]
    )
)

lead_idx = [lead_map.get(pair) for pair in gmv_pairs]

mask = pd.Series(
    [idx is not None for idx in lead_idx],
    index=df_gmv.index
)

# GMV đã match
matched = df_gmv.loc[mask].copy()

# Lead tương ứng
matched_leads = df_leads.loc[
    [idx for idx in lead_idx if idx is not None]
].copy()

# Loại khỏi GMV
df_gmv = df_gmv.loc[~mask].copy()

print("Matched:", len(matched))
print("Matched leads:", len(matched_leads))
print("Remaining GMV:", len(df_gmv))

Matched: 2
Matched leads: 2
Remaining GMV: 3


In [ ]:
df_gmv[["Sales_infor", "Type", "Pay Time"]]

,Sales_infor,Type,Pay Time
253,HN2 - HuongVHT,Other,2026-05-22
304,HN2 - HienVTK,Lives,2026-05-25
462,HN2 - HuongVHT,Other,2026-05-31


In [ ]:
df_gmv["Nguồn"]

253           NaN
304    Tiktokshop
462           NaN
Name: Nguồn, dtype: object

In [ ]:
df_gmv_test = pd.read_csv("./cleaned_data/gmv_clean_sales_infor.csv", encoding='utf-8-sig')
df_filtered_test = df_gmv_test[df_gmv_test['Nguồn'].notna()]

In [ ]:
df_filtered_test[["Type", "Nguồn"]]

,Type,Nguồn
304,Lives,Tiktokshop
366,广告,Partnership
378,广告,Partnership
407,广告,Partnership
457,Lives,Tiktok


In [89]:
df_to_check = pd.read_csv("./output/test_3.csv")

In [91]:
df_to_check["UID_clean"].value_counts()

UID_clean
3312285639    2
3310023176    2
3302992772    1
3311874241    1
3311812212    1
3311784220    1
3288489392    1
3311922591    1
3303143877    1
3302753136    1
3311918781    1
3311993190    1
3312097348    1
3270105568    1
3312014154    1
3302670521    1
3297314626    1
3268339546    1
3311490911    1
3312164356    1
3312172635    1
3312248783    1
3175938767    1
3271847872    1
3179205818    1
3310392292    1
3311423632    1
3287789579    1
3299808858    1
3267891426    1
3311032514    1
3298993560    1
3311153969    1
3311199404    1
3280414186    1
3311099197    1
3292493174    1
3311538332    1
3310486815    1
3311555589    1
3302606907    1
3311508463    1
3303488588    1
3304138773    1
3311450402    1
3274632378    1
3311727959    1
3294667215    1
Name: count, dtype: int64

#### 2 đơn thuộc về HN2 - HuongVHT trong gmv thực chất là cháu của sale
#### Còn lại 1 đơn có 1 uid nhưng 2 sdt có uid là 3310023176 thì số điện thoại đúng là 82-1029349211
#### sau khi check 1 số đơn thì có xuất hiện "Sales" tự tìm kiếm